In [ ]:
# ==========================================
# 1. Import Custom Modules & Environment
# ==========================================
import os
import sys
import torch
from ultralytics import YOLO

sys.path.append(os.path.abspath('..'))
import attention
import custom_loss
import models_init
from ultralytics.nn import tasks

# Register custom modules so Ultralytics can parse yolo-custom.yaml
models_init.register_custom_modules()

# Hook WiseIoU v3 loss into the training loop
tasks.v8DetectionLoss = custom_loss.CustomDetectionLoss

print("Successfully loaded attention mechanisms and custom loss extensions!")

# ==========================================
# 2. Build Fully-Upgraded Architecture Model
# ==========================================
# Loads custom YAML containing BiFPN, CBAM, and P2 head, transferring COCO weights
model = YOLO('../yolo-custom.yaml').load('yolov8n.pt')
print("Successfully initialized Full Tier 1+2+3 Custom YOLOv8 Architecture.")

# ==========================================
# 3. Pre-training Sanity Checks
# ==========================================
print(f"Model depth: {len(model.model)} layers")
print(f"Detection head: {type(model.model[-1]).__name__}")
with torch.no_grad():
    dummy = torch.randn(1, 3, 640, 640)
    out = model.model(dummy)
    if isinstance(out, (list, tuple)):
        for i, item in enumerate(out):
            if isinstance(item, dict):
                print(f"Output[{i}] dict keys: {list(item.keys())}")
                for k, v in item.items():
                    if hasattr(v, 'shape'):
                        print(f"  {k}: {v.shape}")
            elif hasattr(item, 'shape'):
                print(f"Output[{i}] shape: {item.shape}")
    elif hasattr(out, 'shape'):
        print(f"Output shape: {out.shape}")
print("Pre-training validation passed.")

# ==========================================
# 4. Train with Complete Advanced Hyperparameters
# ==========================================
results = model.train(
    data='../dataset_enhanced.yaml',     # Preprocessed dataset path
    epochs=150,                          # Maximum training epochs
    patience=35,                         # Early stopping patience
    imgsz=640,                           # Base image size
    batch=16,                            # Batch size fitting GPU memory
    seed=42,                             # Reproducible baseline
    
    # Optimizer & Learning Rate (Tier 1 & Tier 2)
    optimizer='AdamW',                   
    lr0=0.0025,                          # Conservative starting LR for complex architecture
    cos_lr=True,                         # Cosine LR scheduling
    amp=True,                            # Automatic Mixed Precision
    ema=True,                            # Exponential Moving Average for weight stability
    
    # Multi-Scale Training (Tier 2 feature)
    multi_scale=0.5,                     # Sample image sizes from 320 to 960 (with imgsz=640)
    
    # Advanced Augmentations (Tier 3: Mixup for complex scene blending)
    mixup=0.1,                           # Light mixup probability
    
    # Preprocessing Protection (Tier 1: Disable conflicting color jitters)
    hsv_h=0.0,                           # No hue shifting (protects Grayworld color balance)
    hsv_s=0.02,                          # Minimal saturation variance
    hsv_v=0.02,                          # Minimal brightness variance
    mosaic=1.0,                          # Spatial mosaic augmentation enabled
    fliplr=0.5,                          # Horizontal flipping enabled
    
    name='underwater_custom_v1'
)

print("Training session finished successfully! Model saved at: runs/detect/underwater_custom_v1/weights/best.pt")